# OLAP Mastery Challenge + Capstone (couche Gold)

**Cours :** Big Data Storage & Management — M2 HSIM, IAE Lyon

Ce notebook contient :
1. **La séance OLAP** : connexion à PostgreSQL (Aiven), setup du schéma en flocon du cours, requêtes **T1 à T10**.
2. **Le bonus capstone** : le schéma Gold de notre entrepôt (consommation d'antibiotiques en ville, Open Medic + INSEE) et **3 requêtes OLAP** correspondant à nos KPI.

## 0. Connexion à PostgreSQL (Aiven)
Renseigner les informations de *Overview → Connection information* sur Aiven. Le mot de passe est demandé à l'exécution : il n'est jamais écrit dans le notebook.

In [ ]:
!pip -q install psycopg2-binary

import psycopg2, pandas as pd
from getpass import getpass
from IPython.display import display

DB_HOST = "pg-xxxxxxxx.aivencloud.com"   # Host
DB_PORT = 12345                          # Port (nombre, sans guillemets)
DB_NAME = "defaultdb"
DB_USER = "avnadmin"

DB_PASSWORD = getpass("Mot de passe Aiven : ")
conn = psycopg2.connect(host=DB_HOST, port=DB_PORT, dbname=DB_NAME, user=DB_USER,
                        password=DB_PASSWORD, sslmode="require")

def run_sql(sql, show=True):
    """Exécute du SQL ; renvoie un DataFrame si la requête produit des lignes."""
    with conn.cursor() as cur:
        try:
            cur.execute(sql)
        except Exception:
            conn.rollback()
            raise
        df = None
        if cur.description:
            df = pd.DataFrame(cur.fetchall(), columns=[d[0] for d in cur.description])
    conn.commit()
    if df is not None and show:
        display(df)
    return df

run_sql("SELECT version()")

## 1. Setup du challenge OLAP (schéma en flocon du cours + données)

In [ ]:
# Run the full setup (schema + seed) on your Aurora/Postgres
_ = run_sql("""
-- Clean start (drop in FK-safe order)
DROP TABLE IF EXISTS fact_sales       CASCADE;
DROP TABLE IF EXISTS dim_product      CASCADE;
DROP TABLE IF EXISTS dim_subcategory  CASCADE;
DROP TABLE IF EXISTS dim_category     CASCADE;
DROP TABLE IF EXISTS dim_time         CASCADE;
DROP TABLE IF EXISTS dim_region       CASCADE;
DROP TABLE IF EXISTS dim_city         CASCADE;
DROP TABLE IF EXISTS dim_state        CASCADE;
DROP TABLE IF EXISTS dim_country      CASCADE;
DROP TABLE IF EXISTS dim_channel      CASCADE;
DROP TABLE IF EXISTS dim_subchannel   CASCADE;

-- Dimensions
CREATE TABLE dim_category (
  category_id   BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
  category_name TEXT NOT NULL UNIQUE
);

CREATE TABLE dim_subcategory (
  subcategory_id   BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
  category_id      BIGINT NOT NULL REFERENCES dim_category(category_id) ON UPDATE CASCADE ON DELETE RESTRICT,
  subcategory_name TEXT  NOT NULL
);

CREATE TABLE dim_product (
  product_id     BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
  subcategory_id BIGINT NOT NULL REFERENCES dim_subcategory(subcategory_id) ON UPDATE CASCADE ON DELETE RESTRICT,
  name           TEXT   NOT NULL
);

CREATE TABLE dim_time (
  time_id BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
  month   INT    NOT NULL CHECK (month BETWEEN 1 AND 12),
  quarter TEXT   NOT NULL CHECK (quarter IN ('Q1','Q2','Q3','Q4')),
  year    INT    NOT NULL CHECK (year >= 1900)
);

CREATE TABLE dim_country (
  country_id   BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
  country_name TEXT NOT NULL UNIQUE
);

CREATE TABLE dim_state (
  state_id   BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
  country_id BIGINT NOT NULL REFERENCES dim_country(country_id) ON UPDATE CASCADE ON DELETE RESTRICT,
  state_name TEXT   NOT NULL
);

CREATE TABLE dim_city (
  city_id  BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
  state_id BIGINT NOT NULL REFERENCES dim_state(state_id) ON UPDATE CASCADE ON DELETE RESTRICT,
  city_name TEXT  NOT NULL
);

CREATE TABLE dim_region (
  region_id   BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
  city_id     BIGINT NOT NULL REFERENCES dim_city(city_id) ON UPDATE CASCADE ON DELETE RESTRICT,
  region_name TEXT   NOT NULL
);

CREATE TABLE dim_subchannel (
  subchannel_id   BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
  subchannel_name TEXT NOT NULL
);

CREATE TABLE dim_channel (
  channel_id    BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
  subchannel_id BIGINT REFERENCES dim_subchannel(subchannel_id) ON UPDATE CASCADE ON DELETE SET NULL,
  channel_name  TEXT NOT NULL
);

-- Fact
CREATE TABLE fact_sales (
  sale_id   BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
  product_id BIGINT NOT NULL REFERENCES dim_product(product_id) ON UPDATE CASCADE ON DELETE RESTRICT,
  time_id    BIGINT NOT NULL REFERENCES dim_time(time_id)       ON UPDATE CASCADE ON DELETE RESTRICT,
  region_id  BIGINT NOT NULL REFERENCES dim_region(region_id)   ON UPDATE CASCADE ON DELETE RESTRICT,
  channel_id BIGINT NOT NULL REFERENCES dim_channel(channel_id) ON UPDATE CASCADE ON DELETE RESTRICT,
  quantity   INT,
  revenue    NUMERIC(10,2)
);

CREATE INDEX ON fact_sales (time_id);
CREATE INDEX ON fact_sales (region_id);
CREATE INDEX ON fact_sales (product_id);
CREATE INDEX ON fact_sales (channel_id);

-- Seed (synthetic)
INSERT INTO dim_category (category_name) VALUES ('Apparel'), ('Home');
INSERT INTO dim_subcategory (category_id, subcategory_name) VALUES (1,'T-Shirts'), (2,'Mugs');
INSERT INTO dim_product (subcategory_id, name) VALUES
  (1,'T-Shirt Red L'), (1,'T-Shirt Blue L'), (2,'Ceramic Mug');

-- Two years to enable time-based challenges
INSERT INTO dim_time (month, quarter, year) VALUES
  (1,'Q1',2025),(2,'Q1',2025),(3,'Q1',2025),(4,'Q2',2025),(5,'Q2',2025),(6,'Q2',2025),
  (7,'Q3',2025),(8,'Q3',2025),(9,'Q3',2025),(10,'Q4',2025),(11,'Q4',2025),(12,'Q4',2025),
  (1,'Q1',2026),(2,'Q1',2026),(3,'Q1',2026),(4,'Q2',2026);

INSERT INTO dim_country (country_name) VALUES ('France'),('Spain');
INSERT INTO dim_state (country_id, state_name) VALUES (1,'Île-de-France'),(2,'Catalonia');
INSERT INTO dim_city (state_id, city_name) VALUES (1,'Paris'),(2,'Barcelona');
INSERT INTO dim_region (city_id, region_name) VALUES (1,'North Europe'),(2,'South Europe');

INSERT INTO dim_subchannel (subchannel_name) VALUES ('General');
INSERT INTO dim_channel (channel_name, subchannel_id) VALUES ('Online',1), ('Store',1);

-- Minimal sales spread across regions, time, and channels
INSERT INTO fact_sales (product_id,time_id,region_id,channel_id,quantity,revenue) VALUES
  -- 2025 Q1/Q2
  (1,1,1,1,10,199.90),(2,4,1,2,15,329.90),(3,5,2,1,5,64.90),
  -- 2025 Q3/Q4
  (1,7,2,2,12,239.88),(2,10,1,1,20,439.80),(3,12,2,2, 8,103.92),
  -- 2026 Q1/Q2 (partial)
  (1,13,1,1, 9,179.91),(2,14,2,1, 5,109.95),(3,16,1,2, 7, 90.93);
""", show=False)
print("✅ Schema created and data seeded.")

### T1 — GROUPING SETS Multilevel Summary
Trois niveaux de détail en une requête : (année, trimestre, région), (année, région), (année). `NULLS LAST` place les sous-totaux après le détail.

In [ ]:
sql = """
SELECT dt.year, dt.quarter, dr.region_name, SUM(fs.revenue) AS revenue
FROM fact_sales fs
JOIN dim_time   dt ON fs.time_id   = dt.time_id
JOIN dim_region dr ON fs.region_id = dr.region_id
GROUP BY GROUPING SETS ((dt.year, dt.quarter, dr.region_name), (dt.year, dr.region_name), (dt.year))
ORDER BY dt.year, dt.quarter NULLS LAST, dr.region_name NULLS LAST
"""
run_sql(sql)

### T2 — ROLLUP by Quarter and Category
`ROLLUP (quarter, category)` produit le détail, le sous-total par trimestre et le total général.

In [ ]:
sql = """
SELECT dt.quarter, dc.category_name, SUM(fs.revenue) AS revenue
FROM fact_sales fs
JOIN dim_time        dt ON fs.time_id        = dt.time_id
JOIN dim_product     dp ON fs.product_id     = dp.product_id
JOIN dim_subcategory ds ON dp.subcategory_id = ds.subcategory_id
JOIN dim_category    dc ON ds.category_id    = dc.category_id
GROUP BY ROLLUP (dt.quarter, dc.category_name)
ORDER BY dt.quarter NULLS LAST, dc.category_name NULLS LAST
"""
run_sql(sql)

### T3 — CUBE for Region x Channel
`CUBE (region, channel)` produit toutes les combinaisons : détail, sous-totaux par région, par canal, et total général.

In [ ]:
sql = """
SELECT dr.region_name, dch.channel_name, SUM(fs.revenue) AS revenue
FROM fact_sales fs
JOIN dim_region  dr  ON fs.region_id  = dr.region_id
JOIN dim_channel dch ON fs.channel_id = dch.channel_id
GROUP BY CUBE (dr.region_name, dch.channel_name)
ORDER BY dr.region_name NULLS LAST, dch.channel_name NULLS LAST
"""
run_sql(sql)

### T4 — Top-3 Products per Region-Quarter
Chiffre d'affaires par (région, année, trimestre, produit), puis `DENSE_RANK()` dans chaque groupe et filtre sur le rang.

In [ ]:
sql = """
WITH agg AS (
  SELECT dr.region_name, dt.year, dt.quarter, dp.name AS product_name, SUM(fs.revenue) AS revenue
  FROM fact_sales fs
  JOIN dim_region  dr ON fs.region_id  = dr.region_id
  JOIN dim_time    dt ON fs.time_id    = dt.time_id
  JOIN dim_product dp ON fs.product_id = dp.product_id
  GROUP BY dr.region_name, dt.year, dt.quarter, dp.name
),
ranked AS (
  SELECT *, DENSE_RANK() OVER (PARTITION BY region_name, year, quarter ORDER BY revenue DESC) AS rnk
  FROM agg
)
SELECT region_name, year, quarter, product_name, revenue, rnk
FROM ranked
WHERE rnk <= 3
ORDER BY region_name, year, quarter, rnk
"""
run_sql(sql)

### T5 — Quarter-over-Quarter Trend
`LAG(revenue)` récupère le trimestre précédent dans chaque série (région, catégorie) ; `NULLIF` évite la division par zéro.

In [ ]:
sql = """
WITH agg AS (
  SELECT dr.region_name, dc.category_name, dt.year, dt.quarter, SUM(fs.revenue) AS revenue
  FROM fact_sales fs
  JOIN dim_region      dr ON fs.region_id      = dr.region_id
  JOIN dim_time        dt ON fs.time_id        = dt.time_id
  JOIN dim_product     dp ON fs.product_id     = dp.product_id
  JOIN dim_subcategory ds ON dp.subcategory_id = ds.subcategory_id
  JOIN dim_category    dc ON ds.category_id    = dc.category_id
  GROUP BY dr.region_name, dc.category_name, dt.year, dt.quarter
)
SELECT region_name, category_name, year, quarter, revenue,
       LAG(revenue) OVER w AS prev_revenue,
       revenue - LAG(revenue) OVER w AS qoq_change,
       ROUND(100.0 * (revenue - LAG(revenue) OVER w) / NULLIF(LAG(revenue) OVER w, 0), 2) AS qoq_pct
FROM agg
WINDOW w AS (PARTITION BY region_name, category_name ORDER BY year, quarter)
ORDER BY region_name, category_name, year, quarter
"""
run_sql(sql)

### T6 — YTD (Year-to-Date) by Category
Cumul depuis janvier : `SUM(...) OVER (PARTITION BY year, category ORDER BY month ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)`.

In [ ]:
sql = """
WITH agg AS (
  SELECT dt.year, dc.category_name, dt.month, SUM(fs.revenue) AS monthly_revenue
  FROM fact_sales fs
  JOIN dim_time        dt ON fs.time_id        = dt.time_id
  JOIN dim_product     dp ON fs.product_id     = dp.product_id
  JOIN dim_subcategory ds ON dp.subcategory_id = ds.subcategory_id
  JOIN dim_category    dc ON ds.category_id    = dc.category_id
  GROUP BY dt.year, dc.category_name, dt.month
)
SELECT year, category_name, month, monthly_revenue,
       SUM(monthly_revenue) OVER (PARTITION BY year, category_name ORDER BY month
                                  ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS ytd_revenue
FROM agg
ORDER BY year, category_name, month
"""
run_sql(sql)

### T7 — Share of Wallet (Category Share)
Part de chaque catégorie : chiffre de la catégorie / total du couple (région, trimestre), calculé avec une fonction de fenêtre.

In [ ]:
sql = """
WITH agg AS (
  SELECT dr.region_name, dt.year, dt.quarter, dc.category_name, SUM(fs.revenue) AS category_rev
  FROM fact_sales fs
  JOIN dim_region      dr ON fs.region_id      = dr.region_id
  JOIN dim_time        dt ON fs.time_id        = dt.time_id
  JOIN dim_product     dp ON fs.product_id     = dp.product_id
  JOIN dim_subcategory ds ON dp.subcategory_id = ds.subcategory_id
  JOIN dim_category    dc ON ds.category_id    = dc.category_id
  GROUP BY dr.region_name, dt.year, dt.quarter, dc.category_name
)
SELECT region_name, year, quarter, category_name, category_rev,
       SUM(category_rev) OVER (PARTITION BY region_name, year, quarter) AS total_rev,
       ROUND(category_rev / SUM(category_rev) OVER (PARTITION BY region_name, year, quarter), 4) AS share
FROM agg
ORDER BY region_name, year, quarter, share DESC
"""
run_sql(sql)

### T8 — Drill Across (Channel Mix, 2025)
Filtre sur 2025 avant l'agrégation (`WHERE`), puis part de chaque canal dans son trimestre.

In [ ]:
sql = """
WITH agg AS (
  SELECT dt.quarter, dch.channel_name, SUM(fs.revenue) AS revenue
  FROM fact_sales fs
  JOIN dim_time    dt  ON fs.time_id    = dt.time_id
  JOIN dim_channel dch ON fs.channel_id = dch.channel_id
  WHERE dt.year = 2025
  GROUP BY dt.quarter, dch.channel_name
)
SELECT quarter, channel_name, revenue,
       SUM(revenue) OVER (PARTITION BY quarter) AS quarter_total,
       ROUND(100.0 * revenue / SUM(revenue) OVER (PARTITION BY quarter), 2) AS pct_of_quarter
FROM agg
ORDER BY quarter, pct_of_quarter DESC
"""
run_sql(sql)

### T9 — Detect Declines (QoQ)
Série trimestrielle par (région, catégorie), `LAG`, puis `DISTINCT` sur les couples ayant connu au moins une baisse.

In [ ]:
sql = """
WITH agg AS (
  SELECT dr.region_name, dc.category_name, dt.year, dt.quarter, SUM(fs.revenue) AS revenue
  FROM fact_sales fs
  JOIN dim_region      dr ON fs.region_id      = dr.region_id
  JOIN dim_time        dt ON fs.time_id        = dt.time_id
  JOIN dim_product     dp ON fs.product_id     = dp.product_id
  JOIN dim_subcategory ds ON dp.subcategory_id = ds.subcategory_id
  JOIN dim_category    dc ON ds.category_id    = dc.category_id
  GROUP BY dr.region_name, dc.category_name, dt.year, dt.quarter
),
series AS (
  SELECT *, LAG(revenue) OVER (PARTITION BY region_name, category_name ORDER BY year, quarter) AS prev_revenue
  FROM agg
)
SELECT DISTINCT region_name, category_name
FROM series
WHERE revenue < prev_revenue
ORDER BY region_name, category_name
"""
run_sql(sql)

### T10 — Dense Rollup + Filter
`ROLLUP (year, category)` puis `HAVING SUM(revenue) > 200` : le filtre porte sur des totaux, d'où `HAVING` et non `WHERE`.

In [ ]:
sql = """
SELECT dt.year, dc.category_name, SUM(fs.revenue) AS revenue
FROM fact_sales fs
JOIN dim_time        dt ON fs.time_id        = dt.time_id
JOIN dim_product     dp ON fs.product_id     = dp.product_id
JOIN dim_subcategory ds ON dp.subcategory_id = ds.subcategory_id
JOIN dim_category    dc ON ds.category_id    = dc.category_id
GROUP BY ROLLUP (dt.year, dc.category_name)
HAVING SUM(fs.revenue) > 200
ORDER BY dt.year NULLS LAST, dc.category_name NULLS LAST
"""
run_sql(sql)

## 2. Bonus capstone — notre entrepôt (couche Gold) chargé avec les données réelles

**Sujet :** consommation d'antibiotiques en ville (ODD 3). **Sources :** Open Medic 2024 et 2025 (Assurance Maladie) et estimations de population (INSEE).

**Constellation de faits** : `fact_dispensation` (délivrances) et `fact_population` partagent les dimensions Temps, Région, Tranche d'âge et Sexe. La dimension Médicament est **en flocon** selon la hiérarchie ATC (CIP13 → ATC5 → ATC4 → ATC3 → ATC2 → ATC1).

La cellule suivante récupère le dépôt GitHub du projet, puis lance `pipeline/03_load_gold.py` : il crée le schéma `capstone` (séparé des tables du challenge) et le remplit avec les fichiers Silver du dépôt (258 116 délivrances d'antibiotiques et la population INSEE correspondante).

In [ ]:
import os
!rm -rf capstone-bdsm && git clone -q https://github.com/mejineanaikaj-png/capstone-bdsm.git
os.environ.update(PGHOST=DB_HOST, PGPORT=str(DB_PORT), PGDATABASE=DB_NAME,
                  PGUSER=DB_USER, PGPASSWORD=DB_PASSWORD, PGSSLMODE="require")
!python capstone-bdsm/pipeline/03_load_gold.py

### Requête A — KPI 1 et KPI 2 : taux pour 1 000 habitants et dépense par habitant
**Question :** combien de boîtes d'antibiotiques sont délivrées pour 1 000 habitants, et combien l'Assurance Maladie rembourse-t-elle par habitant, dans chaque région et chaque année ?

**Opérations OLAP :** slice (ATC2 = J01), roll-up (CIP13 → ATC2 ; âge et sexe → région), drill-across entre les deux faits. Chaque fait est agrégé **avant** la jointure pour ne pas multiplier les lignes.

In [ ]:
sql = """
SET search_path TO capstone;
WITH antibio AS (
  SELECT f.time_id, f.region_id,
         SUM(f.boxes)             AS boxes,
         SUM(f.amount_reimbursed) AS amount
  FROM fact_dispensation f
  JOIN dim_product p ON p.product_id = f.product_id
  JOIN dim_atc5   a5 ON a5.atc5_code = p.atc5_code
  JOIN dim_atc4   a4 ON a4.atc4_code = a5.atc4_code
  JOIN dim_atc3   a3 ON a3.atc3_code = a4.atc3_code
  WHERE a3.atc2_code = 'J01'
  GROUP BY f.time_id, f.region_id
),
pop AS (
  SELECT time_id, region_id, SUM(population) AS population
  FROM fact_population
  GROUP BY time_id, region_id
)
SELECT t.year, r.region_name,
       a.boxes, p.population,
       ROUND(1000.0 * a.boxes / p.population, 1) AS boxes_per_1000_inhab,
       ROUND(a.amount / p.population, 2)         AS reimbursed_eur_per_inhab
FROM antibio a
JOIN pop        p USING (time_id, region_id)
JOIN dim_time   t ON t.time_id   = a.time_id
JOIN dim_region r ON r.region_id = a.region_id
ORDER BY t.year, boxes_per_1000_inhab DESC
"""
run_sql(sql)

### Requête B — KPI 3 et KPI 6 : évolution annuelle et écart entre régions
**Question :** la consommation baisse-t-elle d'une année sur l'autre, et quel est l'écart entre la région la plus et la moins consommatrice ?

**Opérations OLAP :** même cube que la requête A, puis `LAG` sur l'année et `MAX − MIN` sur une fenêtre partitionnée par année.

In [ ]:
sql = """
SET search_path TO capstone;
WITH antibio AS (
  SELECT f.time_id, f.region_id, SUM(f.boxes) AS boxes
  FROM fact_dispensation f
  JOIN dim_product p ON p.product_id = f.product_id
  JOIN dim_atc5   a5 ON a5.atc5_code = p.atc5_code
  JOIN dim_atc4   a4 ON a4.atc4_code = a5.atc4_code
  JOIN dim_atc3   a3 ON a3.atc3_code = a4.atc3_code
  WHERE a3.atc2_code = 'J01'
  GROUP BY f.time_id, f.region_id
),
pop AS (
  SELECT time_id, region_id, SUM(population) AS population
  FROM fact_population GROUP BY time_id, region_id
),
rate AS (
  SELECT t.year, r.region_name,
         1000.0 * a.boxes / p.population AS rate
  FROM antibio a
  JOIN pop        p USING (time_id, region_id)
  JOIN dim_time   t ON t.time_id   = a.time_id
  JOIN dim_region r ON r.region_id = a.region_id
)
SELECT year, region_name,
       ROUND(rate, 1) AS boxes_per_1000_inhab,
       ROUND(100.0 * (rate - LAG(rate) OVER (PARTITION BY region_name ORDER BY year))
             / LAG(rate) OVER (PARTITION BY region_name ORDER BY year), 1) AS yoy_change_pct,
       ROUND(MAX(rate) OVER (PARTITION BY year) - MIN(rate) OVER (PARTITION BY year), 1) AS gap_max_min
FROM rate
ORDER BY region_name, year
"""
run_sql(sql)

### Requête C — KPI 5 : répartition par spécialité du prescripteur et par âge (2025)
**Question :** quelles spécialités prescrivent le plus d'antibiotiques, et pour quelles tranches d'âge ?

**Opérations OLAP :** dice (année = 2025, ATC2 = J01), `ROLLUP (spécialité, âge)` pour les sous-totaux, part de chaque ligne dans le total national. Les spécialités pesant moins de 1 % sont regroupées en « Autres ».

In [ ]:
sql = """
SET search_path TO capstone;
WITH antibio_2025 AS (
  SELECT pr.specialty_label, ag.age_band_label, f.boxes
  FROM fact_dispensation f
  JOIN dim_time       t  ON t.time_id        = f.time_id
  JOIN dim_prescriber pr ON pr.prescriber_id = f.prescriber_id
  JOIN dim_age_band   ag ON ag.age_band_id   = f.age_band_id
  JOIN dim_product    p  ON p.product_id     = f.product_id
  JOIN dim_atc5       a5 ON a5.atc5_code     = p.atc5_code
  JOIN dim_atc4       a4 ON a4.atc4_code     = a5.atc4_code
  JOIN dim_atc3       a3 ON a3.atc3_code     = a4.atc3_code
  WHERE t.year = 2025 AND a3.atc2_code = 'J01'
),
grouped AS (
  SELECT CASE WHEN SUM(boxes) OVER (PARTITION BY specialty_label)
                   < 0.01 * SUM(boxes) OVER () THEN 'Autres (< 1 % chacune)'
              ELSE specialty_label END AS specialty,
         age_band_label, boxes
  FROM antibio_2025
)
SELECT specialty, age_band_label,
       SUM(boxes) AS boxes,
       ROUND(100.0 * SUM(boxes) / (SELECT SUM(boxes) FROM grouped), 1) AS pct_of_national
FROM grouped
GROUP BY ROLLUP (specialty, age_band_label)
ORDER BY GROUPING(specialty),
         SUM(SUM(boxes)) OVER (PARTITION BY specialty) DESC, specialty,
         GROUPING(age_band_label) DESC, SUM(boxes) DESC
"""
run_sql(sql)